# Who Is Really Leading the Green-Energy Transition?

This combined notebook preserves the original analysis cells from the source notebooks listed below. Cached outputs were removed to keep this publication workspace lightweight; run the relevant cells with the original data files to regenerate charts.

## Source notebooks
- [Global Green Energy Analysis](https://github.com/accidentalscientist/daily_data_analytics_july2025/blob/main/week03_day05/global_green_part1.ipynb)


---

## Source section: Global Green Energy Analysis


# Global Green Energy Analysis 🌱

This and the following week, we'll be doing a large dive into global renewable enregy. This is Part 1 of a massive 10 part series. 
We'll ask questions like:
 - Who are the Green Energy Drivers ? 
 - Is Green Energy really growing ? 
 - How Does Development and Renewables intersect ? Is Green Energy a by-product of wealth ? 
 - If we look at a random set of countries, what does the fossil fuel record look like ? 
 - Which Continents are leading the charge ? 

We'll be diving into the data, visualising the problem and coming up with some answers to these problems. 

This is Part 1 of a massive 10 part series. 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# load in data
raw_data = pd.read_csv('../data/global_sustainable_energy_data.csv')

In [ ]:
raw_data.sample(4)
raw_data.info()

# data looks good, let's visuailise it. 
df = raw_data.copy()

## First Look at the Data

Two key observations emerge from our initial exploration:
- **Data quality looks solid** - clean structure with country-year observations and our target renewable electricity variable is well-populated across the dataset
- **Scale varies dramatically** - the raw numbers hint at massive differences between countries, which makes sense given population and economic disparities, but suggests we'll need smart normalization approaches later

Given the quality of this data, we actually don't need to do any adjustments just yet, but may make some small ones for visualisation purposes. 

In [ ]:
# only look at green energy data
df_clean = df.dropna(subset=['Electricity from renewables (TWh)'])

# get latest year and top 10 countries
latest = df_clean.sort_values('Year').groupby('Entity').tail(1)
top10 = latest.sort_values("Electricity from renewables (TWh)", ascending=False).head(10)

# plot it
sns.set(style="darkgrid")
plt.figure(figsize=(12, 6))
ax = sns.barplot(
    data=top10,
    x="Electricity from renewables (TWh)",
    y="Entity",
    palette=sns.color_palette("Greens", n_colors=10)[::-1],  # reverse order
    hue="Entity",
)

# Annotate bars
for i, value in enumerate(top10["Electricity from renewables (TWh)"]):
    ax.text(value + 5, i, f"{value:.1f}", va='center', fontsize=10)

plt.title("Top 10 Countries by Renewable Electricity (Most Recent Year)", fontsize=14)
plt.xlabel("Electricity from Renewables (TWh)")
plt.ylabel("")
plt.tight_layout()
plt.show()

## Initial Insights

Two patterns emerge from the renewable electricity rankings:
- **Scale dominance** - China's massive lead reflects both population size and serious renewable investment, showing how national capacity matters as much as policy commitment
- **Diverse adoption paths** - seeing Brazil and India alongside European leaders suggests the renewable transition isn't limited to wealthy nations - different economic models can drive green energy growth. Although it is worth highlighting that if we visualised this along a per million people basis it would look very differnt, given India has over 10x the population of Germany, in factc Norway might be the runaway leader given it's small population. 

In [ ]:
# how many years does the dataset cover?
plt.figure(figsize=(12, 1.5))
sns.stripplot(x='Year', data=df_clean, orient="h", size=10, color="coral")
plt.yticks([])
plt.title("Timeline of Years in Dataset")
plt.xlabel("Year")
plt.tight_layout()
plt.show()

## Setting Up the Time Dimension

Before diving into trends, let's understand our temporal scope:
- **Coverage matters** - we need sufficient historical data to identify meaningful patterns in renewable adoption trajectories 
- **Context alignment** - the timeframe should capture the modern renewable energy boom to be relevant for current policy discussions

This is just a nice way to visualise the years covered in this datset. 

In [ ]:
df_clean = df.copy()

# Get top 12 countries by renewables in latest year
latest_year = df_clean["Year"].max()

top_entities = (
    df_clean[df_clean["Year"] == latest_year]
    .sort_values("Electricity from renewables (TWh)", ascending=False)
    .head(12)["Entity"]
    .tolist()
)

# Filter to just top 12 entities
df_top = df_clean[df_clean["Entity"].isin(top_entities)]

# Reorder legend by last-year values
order_by_latest = (
    df_top[df_top["Year"] == latest_year]
    .groupby("Entity")["Electricity from renewables (TWh)"]
    .sum()
    .sort_values(ascending=False)
    .index.tolist()
)

# Plot heatmap of trends grouped by cluster
pivot = (
    df_top
    .pivot_table(index="Entity", columns="Year", values="Electricity from renewables (TWh)")
)
pivot_sorted = pivot.fillna(0)


plt.figure(figsize=(14, 8))
sns.heatmap(pivot_sorted, cmap="YlGnBu", cbar_kws={"label": "TWh"})
plt.title("Clustered Renewable Electricity Trends (Top 30 Countries)")
plt.xlabel("Year")
plt.ylabel("Country (Clustered)")
plt.tight_layout()
plt.show()


## Renewables Over Time

Over the timescale of this dataset, this visualisation allows us to show the top 10 contributors and the impact of each year.
- **Chinese Investment** - We can clearly see wthat a massive investment began to pay off around 2010 and has grown exponentially to the world's leading Renewable Energy Producer, whilst the United States shows a mor graudal progression. 
- **Duopoly Overshaddows** - The Duopoly of China-US overshaddows the other leading players, we can see the consistency of Brazil & Canada across the entire timescale, we can also see how Germany and India have grown their share recently.

A useful addition to this analysis would be excluding the Duopoly and running again as they currently make changes in the other nations difficult to dettect. Essentially however, they are nowhere near the two leaderes. 

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# Top 30 countries by latest year
latest_year = df_clean["Year"].max()
top_countries = (
    df_clean[df_clean["Year"] == latest_year]
    .sort_values("Electricity from renewables (TWh)", ascending=False)
    .head(30)["Entity"]
    .tolist()
)

# Filter and pivot
df_top = df_clean[df_clean["Entity"].isin(top_countries)]
pivot = df_top.pivot(index="Entity", columns="Year", values="Electricity from renewables (TWh)").fillna(0)

# Normalize per country
scaler = StandardScaler()
X_scaled = scaler.fit_transform(pivot)

# Run clustering
k = 4  # Change cluster count as needed
kmeans = KMeans(n_clusters=k, random_state=42)
pivot['Cluster'] = kmeans.fit_predict(X_scaled)

# Merge cluster labels back
pivot_no_cluster = pivot.drop(columns=["Cluster"])
pivot_no_cluster["Cluster"] = kmeans.labels_

# Group by cluster and average
cluster_means = pivot_no_cluster.groupby("Cluster").mean()

# Plot average trend per cluster
plt.figure(figsize=(12, 6))
for cluster_id, row in cluster_means.iterrows():
    plt.plot(row.index, row.values, marker='o', label=f"Cluster {cluster_id}")

plt.title("Average Renewable Electricity Trend per Cluster")
plt.xlabel("Year")
plt.ylabel("Avg. Renewable Electricity (TWh)")
plt.legend(title="Cluster")
plt.tight_layout()
plt.show()



## Clustering Reveals Hidden Patterns

The clustering approach unlocks insights that simple rankings miss entirely:
- **Trajectory typing** - each cluster represents a distinct renewable adoption pathway, revealing that countries don't just differ in scale but in their fundamental growth patterns over time
- **Beyond current leaders** - clustering shows us which countries are on similar paths regardless of their current position, helping identify emerging leaders before they reach the top rankings
- **Policy implications** - understanding these different trajectories helps policymakers see which approach their country is following and what lessons they can learn from cluster peers
- **Predictive power** - by identifying which cluster a country belongs to, we can better anticipate future renewable development patterns and set more realistic targets